In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, Descriptors, rdMolDescriptors
from sklearn.feature_selection import VarianceThreshold

import seaborn as sns
import matplotlib.pyplot as plt

---
## Data split
scaffold split basado en Murcko, que tiende a sobreestimar la separación estructural en datasets con series SAR explícitas (Landrum 2024). So split por clustering de Tanimoto

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
from scipy.spatial.distance import pdist
from sklearn.model_selection import BaseCrossValidator

def get_butina_clusters(mols, sim_cutoff=0.6):
    """
    Generate Butina clusters based on Tanimoto similarity.
    
    Args:
        mols: list of RDKit Mol objects
        sim_cutoff: minimum Tanimoto similarity to join a cluster (0.6 is typical)
    
    Returns:
        list of lists: each sublist contains indices of molecules in that cluster
    """
    # Morgan fingerprints (radius=2, 2048 bits)
    fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = [fpg.GetFingerprint(mol) for mol in mols]
    # Condensed distance matrix: 1 - Tanimoto
    dists = pdist(fps, metric='jaccard')
    # Butina clustering: cutoff is distance = 1 - similarity
    clusters = Butina.ClusterData(dists, len(mols), 1 - sim_cutoff, isDistData=True)
    return clusters

def assign_cluster_ids(mols, sim_cutoff=0.6):
    """Return an array of cluster IDs for each molecule."""
    clusters = get_butina_clusters(mols, sim_cutoff)
    cluster_ids = np.zeros(len(mols), dtype=int)
    for cid, cluster_indices in enumerate(clusters):
        for idx in cluster_indices:
            cluster_ids[idx] = cid
    return cluster_ids

In [ ]:
def train_test_split_by_clusters(cluster_ids, test_ratio=0.2, random_state=42):
    """
    Split clusters so that all compounds of a cluster go to train or test.
    Target: test_ratio of total compounds.
    """
    np.random.seed(random_state)
    unique_clusters = np.unique(cluster_ids)
    # Shuffle clusters
    np.random.shuffle(unique_clusters)
    
    # Count compounds per cluster
    cluster_sizes = {cid: np.sum(cluster_ids == cid) for cid in unique_clusters}
    
    test_idx = []
    current_test_size = 0
    target_test_size = int(len(cluster_ids) * test_ratio)
    
    for cid in unique_clusters:
        size = cluster_sizes[cid]
        if current_test_size + size <= target_test_size:
            test_idx.extend(np.where(cluster_ids == cid)[0].tolist())
            current_test_size += size
        else:
            # This cluster goes to train, and all remaining clusters also go to train
            break
    
    train_idx = [i for i in range(len(cluster_ids)) if i not in test_idx]
    
    print(f"Train compounds: {len(train_idx)} ({len(train_idx)/len(cluster_ids):.1%})")
    print(f"Test compounds : {len(test_idx)} ({len(test_idx)/len(cluster_ids):.1%})")
    print(f"Train clusters : {len(set(cluster_ids[train_idx]))}")
    print(f"Test clusters  : {len(set(cluster_ids[test_idx]))}")
    print(f"Overlap clusters? {set(cluster_ids[train_idx]) & set(cluster_ids[test_idx]) == {}}")
    
    return train_idx, test_idx

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import pytorch_lightning as pl

class MoleculeDataModule(pl.LightningDataModule):
    def __init__(self, X, y, cluster_ids, batch_size=32):
        super().__init__()
        self.X = torch.tensor(X, dtype=torch.float32)   # fingerprints
        self.y = torch.tensor(y, dtype=torch.float32)
        self.cluster_ids = cluster_ids
        self.batch_size = batch_size
        self.train_idx = None
        self.val_idx = None

    def setup(self, stage=None):
        # For a single train/val split (e.g., after train_test_split_by_clusters)
        # You would set self.train_idx and self.val_idx from outside.
        # For CV, you would set them inside each fold.
        pass

    def set_split(self, train_idx, val_idx):
        self.train_idx = train_idx
        self.val_idx = val_idx

    def train_dataloader(self):
        return DataLoader(TensorDataset(self.X[self.train_idx], self.y[self.train_idx]),
                          batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(TensorDataset(self.X[self.val_idx], self.y[self.val_idx]),
                          batch_size=self.batch_size)

In [ ]:
# ---- Load your data ----
# df = pd.read_csv('your_data.csv')
# smiles_list = df['SMILES'].tolist()
# activities = df['active'].values  # 0/1

# For demonstration, create dummy data
smiles_list = ["CCO", "CCN", "c1ccccc1", "c1ccccc1N", "c1ccccc1O"] * 500  # 2500 molecules
mols = [Chem.MolFromSmiles(s) for s in smiles_list]
activities = np.random.randint(0, 2, len(mols))

# ---- Step 1: Get clusters ----
cluster_ids = assign_cluster_ids(mols, sim_cutoff=0.6)
print(f"Number of clusters: {len(np.unique(cluster_ids))}")

# ---- Step 2: Generate fingerprints (features) ----
fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
X = np.array([fpg.GetFingerprintAsNumPy(mol) for mol in mols], dtype=np.float32)
y = activities

# ---- Option A: Single train/test split (80/20) ----
train_idx, test_idx = train_test_split_by_clusters(cluster_ids, test_ratio=0.2)
dm = MoleculeDataModule(X, y, cluster_ids, batch_size=32)
dm.set_split(train_idx, test_idx)

# ---- Train a simple Lightning model ----
class MLP(pl.LightningModule):
    def __init__(self, input_dim=2048):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze()
    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.nn.functional.binary_cross_entropy_with_logits(y_hat, y)
        self.log('train_loss', loss)
        return loss
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

model = MLP(input_dim=X.shape[1])
trainer = pl.Trainer(max_epochs=10, accelerator='cpu')
trainer.fit(model, datamodule=dm)

# ---- Option B: 5-fold cross-validation by clusters ----
cluster_kfold = ClusterKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(cluster_kfold.split(X, y, groups=cluster_ids)):
    print(f"\nFold {fold+1}: train {len(train_idx)}, test {len(test_idx)}")
    dm.set_split(train_idx, test_idx)
    trainer.fit(model, datamodule=dm)
    # Evaluate here or store metrics